# Extract Skills from Job Descriptions (JD)

This notebook processes multiple parquet files in batches to avoid memory overflow.

## Key Features:
- ✅ Processes files in batches to avoid OOM errors
- ✅ Progress tracking with tqdm
- ✅ Incremental saving of results
- ✅ Resume capability if process is interrupted
- ✅ Memory-efficient processing

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc
from tqdm.auto import tqdm
from skillner import Pipeline
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - Modify these parameters
# ============================================================================

# Input/Output paths
INPUT_DIR = Path("../JD")  # Directory containing parquet files
OUTPUT_DIR = Path("./output/extracted_skills")  # Output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Processing parameters
BATCH_SIZE = 10  # Number of files to process at once (adjust based on your memory)
ROWS_PER_CHUNK = 1000  # Number of rows to process per chunk within each file

# Column names
TEXT_COLUMN = 'job_description'  # Name of the column containing text
ID_COLUMN = 'job_id'  # Name of the ID column (optional)

# Resume capability
RESUME_FROM_CHECKPOINT = True  # Set to True to skip already processed files
CHECKPOINT_FILE = OUTPUT_DIR / "processed_files.txt"  # Tracks processed files

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE} files")
print(f"Chunk size: {ROWS_PER_CHUNK} rows")

## Helper Functions

In [ ]:
def get_processed_files():
    """Get set of already processed files from checkpoint"""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return set(line.strip() for line in f)
    return set()

def mark_file_as_processed(filename):
    """Add filename to checkpoint file"""
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{filename}\n")

def process_text_batch(texts, nlp):
    """Extract skills from a batch of texts"""
    results = []
    for text in texts:
        if pd.isna(text) or text == '':
            results.append({'skills': [], 'num_skills': 0})
        else:
            try:
                annotations = nlp(text)
                skills = [ent['doc_node_value'] for ent in annotations['results']['full_matches']]
                results.append({'skills': skills, 'num_skills': len(skills)})
            except Exception as e:
                print(f"Error processing text: {e}")
                results.append({'skills': [], 'num_skills': 0})
    return results

def process_parquet_file_in_chunks(filepath, nlp, chunk_size=ROWS_PER_CHUNK):
    """
    Process a single parquet file in chunks to avoid memory overflow.
    
    Parameters
    ----------
    filepath : Path
        Path to the parquet file
    nlp : Pipeline
        SkillNER pipeline
    chunk_size : int
        Number of rows to process at once
    
    Returns
    -------
    pd.DataFrame
        Processed dataframe with extracted skills
    """
    # Read parquet file metadata to get total rows
    pf = pd.read_parquet(filepath)
    total_rows = len(pf)
    
    # Process in chunks
    all_results = []
    
    for start_idx in tqdm(range(0, total_rows, chunk_size), 
                          desc=f"Processing {filepath.name}",
                          leave=False):
        end_idx = min(start_idx + chunk_size, total_rows)
        
        # Get chunk
        chunk = pf.iloc[start_idx:end_idx]
        
        # Extract skills
        texts = chunk[TEXT_COLUMN].tolist()
        skill_results = process_text_batch(texts, nlp)
        
        # Add results to chunk
        chunk['extracted_skills'] = [r['skills'] for r in skill_results]
        chunk['num_skills'] = [r['num_skills'] for r in skill_results]
        
        all_results.append(chunk)
        
        # Clear memory
        del chunk, texts, skill_results
        gc.collect()
    
    # Concatenate all chunks
    result_df = pd.concat(all_results, ignore_index=True)
    
    # Clear memory
    del pf, all_results
    gc.collect()
    
    return result_df

def process_file_batch(files, nlp):
    """
    Process a batch of files.
    
    Parameters
    ----------
    files : list of Path
        List of parquet file paths
    nlp : Pipeline
        SkillNER pipeline
    """
    for filepath in files:
        try:
            # Process file in chunks
            result_df = process_parquet_file_in_chunks(filepath, nlp)
            
            # Save result
            output_path = OUTPUT_DIR / f"{filepath.stem}_processed.parquet"
            result_df.to_parquet(output_path, index=False)
            
            # Mark as processed
            mark_file_as_processed(filepath.name)
            
            print(f"✓ Processed {filepath.name}: {len(result_df)} rows, saved to {output_path.name}")
            
            # Clear memory
            del result_df
            gc.collect()
            
        except Exception as e:
            print(f"✗ Error processing {filepath.name}: {e}")
            continue

print("Helper functions loaded.")

## Initialize SkillNER Pipeline

In [ ]:
# Initialize the SkillNER pipeline
print("Loading SkillNER pipeline...")
nlp = Pipeline.load("en")
print("✓ Pipeline loaded successfully!")

## Discover Parquet Files

In [ ]:
# Get all parquet files
all_parquet_files = sorted(INPUT_DIR.glob("*.parquet"))
print(f"Found {len(all_parquet_files)} parquet files in {INPUT_DIR}")

if len(all_parquet_files) == 0:
    raise FileNotFoundError(f"No parquet files found in {INPUT_DIR}")

# Filter out already processed files if resuming
if RESUME_FROM_CHECKPOINT:
    processed_files = get_processed_files()
    files_to_process = [f for f in all_parquet_files if f.name not in processed_files]
    print(f"Already processed: {len(processed_files)} files")
    print(f"Remaining to process: {len(files_to_process)} files")
else:
    files_to_process = all_parquet_files
    print(f"Will process all {len(files_to_process)} files")

# Show first few files
print("\nFirst 5 files to process:")
for f in files_to_process[:5]:
    print(f"  - {f.name}")

## Process Files in Batches

This is the main processing loop. Files are processed in batches to avoid memory overflow.

In [ ]:
# Process files in batches
total_batches = (len(files_to_process) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\nProcessing {len(files_to_process)} files in {total_batches} batches...\n")
print("="*70)

for batch_idx in range(total_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(files_to_process))
    
    batch_files = files_to_process[start_idx:end_idx]
    
    print(f"\nBatch {batch_idx + 1}/{total_batches}: Processing files {start_idx + 1}-{end_idx}")
    print("-"*70)
    
    # Process this batch
    process_file_batch(batch_files, nlp)
    
    # Force garbage collection after each batch
    gc.collect()
    
    print(f"\n✓ Batch {batch_idx + 1}/{total_batches} completed")
    print("="*70)

print("\n" + "="*70)
print("ALL PROCESSING COMPLETE!")
print("="*70)

## Summary Statistics

In [ ]:
# Get all processed output files
output_files = sorted(OUTPUT_DIR.glob("*_processed.parquet"))

print(f"Total output files: {len(output_files)}")

# Calculate summary statistics (sample a few files to avoid loading everything)
sample_size = min(10, len(output_files))
sample_files = np.random.choice(output_files, sample_size, replace=False)

total_rows_sample = 0
total_skills_sample = 0

for f in sample_files:
    df = pd.read_parquet(f)
    total_rows_sample += len(df)
    total_skills_sample += df['num_skills'].sum()

avg_skills_per_row = total_skills_sample / total_rows_sample if total_rows_sample > 0 else 0

print(f"\nSummary (based on {sample_size} sample files):")
print(f"  Sample rows: {total_rows_sample:,}")
print(f"  Total skills extracted: {total_skills_sample:,}")
print(f"  Average skills per row: {avg_skills_per_row:.2f}")
print(f"\nEstimated total rows: {total_rows_sample * len(output_files) // sample_size:,}")
print(f"Estimated total skills: {total_skills_sample * len(output_files) // sample_size:,}")

## Optional: Combine Results

⚠️ **Warning**: Only run this if you have enough memory! This will load all processed files into memory.

In [ ]:
# OPTIONAL: Combine all results into a single file
# Only run this if you have enough memory!

COMBINE_RESULTS = False  # Set to True to combine all results

if COMBINE_RESULTS:
    print("Combining all results...")
    print("⚠️  This may take a lot of memory!\n")
    
    all_dfs = []
    
    for f in tqdm(output_files, desc="Loading files"):
        df = pd.read_parquet(f)
        all_dfs.append(df)
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    # Save combined file
    combined_path = OUTPUT_DIR / "all_skills_extracted.parquet"
    combined_df.to_parquet(combined_path, index=False)
    
    print(f"\n✓ Combined file saved: {combined_path}")
    print(f"  Total rows: {len(combined_df):,}")
    print(f"  File size: {combined_path.stat().st_size / (1024**3):.2f} GB")
    
    del all_dfs, combined_df
    gc.collect()
else:
    print("Skipping result combination. Set COMBINE_RESULTS=True to combine.")
    print("\nResults are saved as individual processed files in:")
    print(f"  {OUTPUT_DIR}")

## Example: Load and Inspect One Result File

In [ ]:
# Load one result file as example
if len(output_files) > 0:
    example_file = output_files[0]
    df_example = pd.read_parquet(example_file)
    
    print(f"Example file: {example_file.name}")
    print(f"Shape: {df_example.shape}")
    print(f"\nColumns: {df_example.columns.tolist()}")
    print(f"\nFirst few rows:")
    display(df_example.head())
    
    # Show some examples with extracted skills
    df_with_skills = df_example[df_example['num_skills'] > 0]
    if len(df_with_skills) > 0:
        print(f"\nExample rows with extracted skills:")
        for idx, row in df_with_skills.head(3).iterrows():
            print(f"\nRow {idx}:")
            print(f"  Text: {row[TEXT_COLUMN][:200]}...")
            print(f"  Skills: {row['extracted_skills']}")
else:
    print("No output files found. Please run the processing cells first.")

## Memory Usage Check

In [ ]:
# Check current memory usage
import psutil
import os

process = psutil.Process(os.getpid())
mem_info = process.memory_info()

print("Current Memory Usage:")
print(f"  RSS: {mem_info.rss / (1024**3):.2f} GB")
print(f"  VMS: {mem_info.vms / (1024**3):.2f} GB")

# System memory
vm = psutil.virtual_memory()
print(f"\nSystem Memory:")
print(f"  Total: {vm.total / (1024**3):.2f} GB")
print(f"  Available: {vm.available / (1024**3):.2f} GB")
print(f"  Used: {vm.percent}%")

## Tips for Optimization

### If you're still running out of memory:

1. **Reduce BATCH_SIZE**: Process fewer files at once
   ```python
   BATCH_SIZE = 5  # or even 1
   ```

2. **Reduce ROWS_PER_CHUNK**: Process fewer rows per chunk
   ```python
   ROWS_PER_CHUNK = 500  # or even 100
   ```

3. **Select specific columns**: Only read columns you need
   ```python
   df = pd.read_parquet(filepath, columns=[TEXT_COLUMN, ID_COLUMN])
   ```

4. **Use Dask**: For truly massive datasets
   ```python
   import dask.dataframe as dd
   ddf = dd.read_parquet(INPUT_DIR / "*.parquet")
   ```

5. **Process on a machine with more RAM**: Consider cloud instances with high memory

### Resume after interruption:

The notebook automatically tracks processed files. If interrupted, just set:
```python
RESUME_FROM_CHECKPOINT = True
```
and re-run. It will skip already processed files.

### Check progress:

```python
processed = len(get_processed_files())
total = len(all_parquet_files)
print(f"Progress: {processed}/{total} ({100*processed/total:.1f}%)")
```